# Bước 2.2: chuẩn hóa ngôn ngữ, trước → sau

**Mục đích:** xem bước chuẩn hóa (Level 2) đổi `text_clean` thành `text_norm` ra sao: mỗi bước sửa bao nhiêu bình luận, ví dụ trước → sau; token, emoji; từ vựng gom được bao nhiêu, viết tắt nào chưa có trong từ điển, nên đặt dấu thanh kiểu nào. Luật: `docs/2-tien-xu-ly/dac-ta-chuan-hoa.md`. Code: `src/processing/normalize.py`. Cập nhật lần cuối: 2026-11-19.

**Nội dung**

1. Đọc file đã xử lý
2. Số bình luận mỗi bước đã sửa (toàn bộ dữ liệu, từ manifest)
3. Từng bước trước → sau (trên mẫu), và kiểm tra file khớp code
4. Token và emoji
5. Danh sách bình luận trước → sau
6. Từ vựng: độ dài, chữ phổ biến, viết tắt, dấu thanh, từ điển
7. Tổng hợp số liệu

**Đọc file nào:** `data/processed/comments.jsonl` và manifest của nó, do lệnh tầng 2 ghi ra. Chưa có file thì chạy trước, từ gốc repo: `.\.venv\Scripts\python.exe -m src.processing.run` (hướng dẫn tầng 2 mục 3). Notebook không ghi file nào.

In [ ]:
import json
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from helpers import *   # ROOT, đường dẫn dữ liệu, N_SHOW, doc_jsonl, doc_da_xu_ly, doc_bo_may, lo_ra, bao_cao, display, pd
thiet_lap_hien_thi()

from src.processing import PIPELINE_VERSION, chuan_hoa, tach_emoji, tach_tu
from src.processing import clauses as L3   # 2.3 tách mệnh đề
from src.processing import clean as L1     # 2.1 làm sạch
from src.processing import normalize as L2   # 2.2 chuẩn hóa

## 1. Đọc file đã xử lý

In [ ]:
df, man = doc_da_xu_ly(["comment_id", "model", "text_clean", "text_norm", "emojis"])
co_nd = df["text_clean"] != ""   # bình luận có nội dung
KQ = {}                          # con số chính, in ở mục 7
print(f"Bình luận có nội dung: {co_nd.sum():,}")

## 2. Số bình luận mỗi bước đã sửa

**Đọc kết quả:** số của **toàn bộ** dữ liệu, lệnh tầng 2 đếm lúc chạy (manifest). N2 (chữ thường) sửa gần như mọi bình luận có chữ hoa, nên con số lớn là bình thường.

In [ ]:
buoc = {k: v for k, v in man["so_binh_luan_moi_buoc_sua"].items() if k.startswith("N")}
bang_buoc = pd.Series(buoc, name="số bình luận bị sửa").to_frame()
bang_buoc["tỷ lệ (%)"] = (100 * bang_buoc["số bình luận bị sửa"] / man["ra"]["so_dong"]).round(3)
for ten, dong in bang_buoc.iterrows():
    KQ[f"Chuẩn hóa – {ten}"] = int(dong["số bình luận bị sửa"])
co_doi2 = df["text_norm"] != df["text_clean"]
KQ["Chuẩn hóa – có thay đổi"] = int(co_doi2.sum())
display(bang_buoc)
print(f"Có thay đổi sau chuẩn hóa: {co_doi2.sum():,} ({100 * co_doi2.mean():.2f}%)")

## 3. Từng bước trước → sau

**Để làm gì:** đưa các cách viết khác nhau của cùng một ý về một dạng, để việc đếm từ, gán nhãn và học mô hình không bị tách vụn: "Đẹp", "ĐẸP", "đẹppppp" là một chữ; "ko", "k", "hok" đều là "không".

Các ô dưới chạy lại từng bước trên `MAU` bình luận ngẫu nhiên để xem mỗi bước đổi gì. Đặt `MAU = None` để chạy cả bộ (khoảng 1 phút). Số đếm trên toàn bộ dữ liệu nằm ở mục 2.

**`text_norm` chỉ dùng cho máy.** Khi trích dẫn hay đưa cho người đọc, dùng `text_clean`.

| Bước | Làm gì | Ví dụ |
|---|---|---|
| N1 | URL, email, số điện thoại, @tên → token | `https://tgdd.vn/...` → `<url>`, `@nguyenvana` → `<user>` |
| N2 | Chữ thường | `ĐẸP` → `đẹp` |
| N3 | Dấu thanh về một kiểu | `hoà`, `thuỷ`, `khoẻ` → `hòa`, `thủy`, `khỏe` |
| N4 | Tiếng cười, emoticon, ký tự lặp | `kkkkk`, `hahaha` → `haha`; `=)))))` → `=))`; `đẹppppp` → `đẹp`; `!!!` → `!` |
| N5 | Tách dấu câu dính chữ | `đẹp,pin` → `đẹp, pin` |
| N6 | Viết tắt, teencode | `ko` → `không`, `dc` → `được`, `mn` → `mọi người` |
| N7 | Dọn khoảng trắng | |

Emoji **giữ nguyên** trong `text_norm` và được tách thêm ra cột `emojis` (mục 4).

**Vì sao theo thứ tự này:**
- N1 trước N2: thay URL, @tên khi còn nguyên dạng, và để tên người dùng không lọt vào bước đếm từ.
- N4 gom tiếng cười trước khi rút ký tự lặp; nếu không, "kkkkk" thành "k" rồi bị N6 đọc thành "không".
- N6 chạy sau N2–N5, vì từ điển viết tắt viết thường và so khớp từng chữ đã tách khỏi dấu câu.

**Code:** các bước nằm trong `src/processing/normalize.py`, từ điển viết tắt ở `src/processing/resources/abbreviations.tsv`. Mỗi ô code dưới đây gọi một bước, áp lên mọi bình luận, rồi in số bình luận bị sửa và ví dụ trước → sau (hàm `bao_cao` trong `notebooks/helpers.py`).

In [ ]:
MAU = 100_000   # số bình luận ngẫu nhiên để chạy lại từng bước; None = cả bộ
mau = mau_ngau_nhien(df[co_nd], MAU)
chuan = mau["text_clean"].copy()   # chuỗi đang chuẩn hóa, đi qua lần lượt N1 → N7
BAO_CAO = []
print(f"Chạy lại trên {len(mau):,} bình luận | code {PIPELINE_VERSION}")

### 3.1 N1. URL, email, số điện thoại, @tên → token

**Để làm gì:** đường link, số điện thoại, tên tài khoản không nói gì về máy, nhưng mỗi cái một khác nên làm từ vựng phình to. Thay bằng token chung vẫn giữ được thông tin "chỗ này có link / có gọi tên người khác".

**Ô code làm gì:**
- `https://…`, `www.…`, tên miền trần như `tgdd.vn`, `bit.ly/abc` → `<url>`. Dấu câu dính cuối link được trả lại cho câu.
- Email → `<email>`; số di động Việt Nam (`0912 345 678`, `0912.345.678`, `+84…`) → `<phone>`.
- `@handle` → `<user>`. Nhờ vậy tên người dùng khác cũng ra khỏi văn bản dùng để phân tích.

**Giới hạn:** tên hiển thị có dấu cách (`@Nguyễn Văn A`, kiểu reply cũ) chỉ thay được chữ đầu.

In [ ]:
truoc, chuan = chuan, chuan.map(L2.thay_token)
bao_cao("N1. URL, email, số điện thoại, @tên → token", truoc, chuan, ma=mau["comment_id"], ghi_vao=BAO_CAO)

### 3.2 N2. Chữ thường

**Ô code làm gì:** viết thường toàn bộ, kể cả `Đ` → `đ`.

**Lưu ý:** sau bước này "AI" (trí tuệ nhân tạo) và "ai" (người nào) thành một chữ. Vì vậy notebook `3_aspect_keywords.ipynb` (khía cạnh theo từ khóa) chạy trên `text_clean`.

In [ ]:
truoc, chuan = chuan, chuan.map(L2.chu_thuong)
bao_cao("N2. Chữ thường", truoc, chuan, ma=mau["comment_id"], ghi_vao=BAO_CAO)

### 3.3 N3. Dấu thanh về một kiểu

**Để làm gì:** cùng một chữ có hai cách đặt dấu, máy coi là hai chữ khác nhau:
- kiểu cũ: dấu ở nguyên âm đầu, `hòa`, `thủy`, `khỏe`;
- kiểu mới: dấu ở nguyên âm sau, `hoà`, `thuỷ`, `khoẻ`.

**Ô code làm gì:** đưa mọi chữ về kiểu đặt ở `KIEU_DAU` trong `normalize.py` (mặc định kiểu cũ).
- Chỉ đổi chữ kết thúc bằng vần mở `oa`, `oe`, `uy`, vì chỉ ở đó hai kiểu khác nhau.
- `quý`, `quỳ` không đổi: chữ `u` trong "qu" là phụ âm, dấu luôn ở `y`.
- `toàn`, `hoàng`, `thuyền` không đổi: có phụ âm cuối thì hai kiểu như nhau.

**Chọn kiểu nào:** mục 6.4 đếm số chữ viết theo mỗi kiểu. Nếu kiểu mới nhiều hơn hẳn thì đổi `KIEU_DAU = "moi"` và tăng `PIPELINE_VERSION`.

In [ ]:
truoc, chuan = chuan, chuan.map(L2.chuan_dau_thanh)
bao_cao(f"N3. Dấu thanh về kiểu {'cũ (hòa)' if L2.KIEU_DAU == 'cu' else 'mới (hoà)'}", truoc, chuan, ma=mau["comment_id"], ghi_vao=BAO_CAO)

### 3.4 N4. Tiếng cười, emoticon, ký tự lặp

**Ô code làm gì**, theo thứ tự:
1. Tiếng cười `kkk`, `hahaha`, `hihi`, `hjhj`, `hehe`, `kaka` → `haha`. Không gồm `huhu` (khóc).
2. Emoticon đứng riêng (`:)))))`, `=)))`, `:(((`) rút còn tối đa 2 ký tự lặp: `:))`, `=))`, `:((`.
3. Chữ có một ký tự lặp từ 3 lần → 1 lần: `đẹppppp` → `đẹp`, `quáaaaa` → `quá` (`á` và `a` tính là cùng chữ).
4. Dấu câu lặp `!!!`, `???` → một dấu; dấu chấm lửng `..`, `....`, `…` → `...`.

**Không đổi:**
- chữ chỉ gồm một ký tự lặp (`zzz`, `aaaa`): rút thành 1 chữ cái thì N6 sẽ đọc nhầm thành viết tắt;
- chữ lặp 2 lần (`xoong`, `đẹpp`);
- chữ số (`1000000`).

**Mất gì:** độ "kéo dài" là dấu hiệu nhấn mạnh cảm xúc, sau bước này không còn trong `text_norm`. Cần thì lấy lại từ `text_clean`.

In [ ]:
truoc, chuan = chuan, chuan.map(L2.rut_gon_lap)
bao_cao("N4. Tiếng cười, emoticon, ký tự lặp", truoc, chuan, ma=mau["comment_id"], ghi_vao=BAO_CAO)

### 3.5 N5. Tách dấu câu dính chữ

**Ô code làm gì:** thêm dấu cách sau `, ; ! ? . ...` khi chúng dính liền chữ phía sau: `đẹp,pin` → `đẹp, pin`, `ổn...mà` → `ổn... mà`.

**Không đổi:** số (`6.7 inch`, `12,5tr`), viết tắt có chấm giữa hai chữ cái đơn (`v.v`), emoticon đứng riêng như `;d`.

In [ ]:
truoc, chuan = chuan, chuan.map(L2.tach_dau_cau)
bao_cao("N5. Tách dấu câu dính chữ", truoc, chuan, ma=mau["comment_id"], ghi_vao=BAO_CAO)

### 3.6 N6. Viết tắt, teencode

**Ô code làm gì:** thay từng chữ đứng riêng theo từ điển `resources/abbreviations.tsv`: `ko` → `không`, `dc` → `được`, `mn` → `mọi người`, `cx` → `cũng`…
- Chỉ thay chữ đứng riêng, đã bỏ ngoặc và dấu câu hai đầu: `(ko)` → `(không)`, nhưng `kon` không đổi.
- `k`, `kg`, `v`, `ms` đứng sau số là đơn vị nên **không** đổi: `500 k`, `1 kg`, `sạc 9 v`, `50 ms`.
- Emoticon như `:v` không bị đụng tới.

**Cố ý chưa đưa vào từ điển** (nhập nhằng trong bình luận điện thoại): `vs` (với / versus), `bh` (bây giờ / bảo hành), `bn` (bạn / bao nhiêu), đại từ một chữ cái `a`, `e`, `c`, `b`… Danh sách đầy đủ ở đầu file từ điển.

**Thêm hay bỏ mục:** xem bảng ứng viên ở mục 6.3 và số lần gặp từng mục ở mục 6.5, sửa file TSV, tăng `PIPELINE_VERSION`, *Restart* kernel rồi chạy lại.

In [ ]:
TU_DIEN = L2.doc_tu_dien()
print(f"Từ điển viết tắt: {len(TU_DIEN)} mục ({L2.TU_DIEN_VIET_TAT.relative_to(ROOT).as_posix()})")
truoc_n6 = chuan   # giữ lại để mục 6.3, 6.5 tìm viết tắt chưa có trong từ điển
truoc, chuan = chuan, chuan.map(L2.mo_viet_tat)
bao_cao("N6. Viết tắt, teencode", truoc, chuan, ma=mau["comment_id"], ghi_vao=BAO_CAO)

### 3.7 N7. Dọn khoảng trắng, và kiểm tra file khớp code

**Ô code làm gì:**
- Các bước trên có thể để lại hai dấu cách liền nhau hay dấu cách ở đầu/cuối; chạy lại bước B6 của làm sạch để dọn.
- In bảng tổng kết các bước trên mẫu.
- So kết quả chạy lại với `text_norm` trong file. Hai bên phải giống hệt nhau; khác thì file được tạo bằng code cũ, chạy lại lệnh tầng 2.

In [ ]:
truoc, chuan = chuan, chuan.map(L1.chuan_khoang_trang)
bao_cao("N7. Dọn khoảng trắng", truoc, chuan, ma=mau["comment_id"], ghi_vao=BAO_CAO)

co_doi_mau = chuan != mau["text_clean"]
display(pd.DataFrame(BAO_CAO + [{"bước": "Có thay đổi (trên mẫu)", "số bình luận bị sửa": int(co_doi_mau.sum()),
                                 "tỷ lệ (%)": round(100 * co_doi_mau.mean(), 3)}]))
lech = chuan != mau["text_norm"]
if lech.any():
    print(f"⚠ {lech.sum():,} bình luận: chạy lại cho kết quả khác text_norm trong file. Chạy lại lệnh tầng 2.")
else:
    print(f"Chạy lại N1–N7 trên {len(mau):,} bình luận cho kết quả giống hệt text_norm trong file.")

## 4. Token và emoji

**Ô code làm gì** (trên toàn bộ dữ liệu):
- Số bình luận có từng loại token `<url>`, `<email>`, `<phone>`, `<user>` trong `text_norm`.
- Tỷ lệ bình luận có emoji và các emoji gặp nhiều nhất, từ cột `emojis`.

**Về emoji:** emoji ghép tính là một (👨‍👩‍👧, 👍🏻, 🇻🇳, 1️⃣). Ký tự ASCII như `^` và ký hiệu như `°`, `©` không tính là emoji, nên con số hơi khác mục 2.5 của notebook `1_raw_audit`.

In [ ]:
for tk in L2.CAC_TOKEN:
    so = int(df["text_norm"].str.contains(tk, regex=False).sum())
    KQ[f"Bình luận có {tk}"] = so
    print(f"có {tk}: {so:,} bình luận")

so_emoji = df["emojis"].map(len)
KQ["Bình luận có emoji"] = int((so_emoji > 0).sum())
KQ["% bình luận có emoji"] = round(100 * (so_emoji > 0).mean(), 2)
print(f"\nBình luận có emoji: {(so_emoji > 0).sum():,} ({100 * (so_emoji > 0).mean():.2f}%) | tổng số emoji: {so_emoji.sum():,}")
dem_emoji = Counter(e for ds in df["emojis"] for e in ds)
bang_emoji = pd.DataFrame(dem_emoji.most_common(N_SHOW), columns=["emoji", "số lần"])
bang_emoji["% trên tổng số emoji"] = (100 * bang_emoji["số lần"] / max(so_emoji.sum(), 1)).round(2)
display(bang_emoji)

## 5. Danh sách bình luận trước → sau

**5a:** `N_SHOW` bình luận đầu tiên (bỏ bình luận rỗng), đặt `text_clean` cạnh `text_norm`. Xuống dòng hiện là `⏎`.

**5b:** `N_SHOW` bình luận ngẫu nhiên bị chuẩn hóa đổi, để kiểm bằng mắt xem có làm sai nghĩa không. Đổi `random_state` để xem bộ khác.

In [ ]:
print("5a. Bình luận đầu tiên: text_clean → text_norm")
sach = df.loc[co_nd, ["comment_id", "model", "text_clean", "text_norm"]].head(N_SHOW)
display(sach.assign(text_clean=sach["text_clean"].str.replace("\n", " ⏎ ", regex=False),
                    text_norm=sach["text_norm"].str.replace("\n", " ⏎ ", regex=False)))

In [ ]:
print("5b. Bình luận bị chuẩn hóa đổi, chọn ngẫu nhiên: text_clean → text_norm")
doi2 = pd.Series(df.index[co_doi2 & co_nd])
chon = doi2.sample(min(N_SHOW, len(doi2)), random_state=0)
xem = df.loc[chon, ["comment_id", "model", "text_clean", "text_norm"]]
display(xem.assign(text_clean=xem["text_clean"].str.replace("\n", " ⏎ ", regex=False),
                   text_norm=xem["text_norm"].str.replace("\n", " ⏎ ", regex=False)))

## 6. Từ vựng

**Để làm gì:** biết người bình luận dùng từ gì, Level 2 gom từ vựng được bao nhiêu, còn viết tắt nào chưa có trong từ điển, nên đặt dấu thanh theo kiểu nào.

Mục này tách chữ bằng `tach_tu()` của module: lấy chữ, số và token `<url>`, `<user>`…; bỏ dấu câu, emoji, emoticon. Chưa tách từ ghép tiếng Việt: "màn hình" là 2 chữ "màn" và "hình". Vì vậy bảng cụm 2 chữ ở mục 9.2 thường dễ đọc hơn bảng 1 chữ.

### 6.1 Từ vựng trước và sau chuẩn hóa, độ dài bình luận

**Ô code làm gì:**
- So số chữ khác nhau trước chuẩn hóa (`text_clean` viết thường) và sau (`text_norm`). Giảm càng nhiều thì chuẩn hóa gom càng được nhiều cách viết.
- Phân bố số chữ mỗi bình luận sau chuẩn hóa, và % bình luận có từ 3 chữ trở xuống (quá ngắn, thường không nêu khía cạnh).

In [ ]:
dem_norm, dem_cum, so_tu = Counter(), Counter(), []
for s in df.loc[co_nd, "text_norm"]:
    ds = tach_tu(s)
    dem_norm.update(ds)
    dem_cum.update(zip(ds, ds[1:]))
    so_tu.append(len(ds))
so_tu = pd.Series(so_tu, index=df.index[co_nd])
dem_clean = Counter()
for s in df.loc[co_nd, "text_clean"]:
    dem_clean.update(tach_tu(s.lower()))

KQ["Số chữ khác nhau trước / sau chuẩn hóa"] = f"{len(dem_clean):,} / {len(dem_norm):,}"
KQ["Trung vị số chữ / bình luận (text_norm)"] = float(so_tu.median())
KQ["% bình luận có ≤ 3 chữ"] = round(100 * (so_tu <= 3).mean(), 1)
display(pd.DataFrame({"text_clean viết thường": [len(dem_clean), sum(dem_clean.values())],
                      "text_norm": [len(dem_norm), sum(dem_norm.values())]},
                     index=["số chữ khác nhau", "tổng số chữ"]))
display(so_tu.describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_frame("số chữ / bình luận"))
print(f"Bình luận có ≤ 3 chữ: {(so_tu <= 3).sum():,} ({100 * (so_tu <= 3).mean():.1f}%)")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(so_tu.clip(upper=100), bins=50, color="#4472c4")
ax.set_xlabel("số chữ trong một bình luận (từ 100 trở lên gộp vào cột cuối)")
ax.set_ylabel("số bình luận")
ax.set_title("Độ dài bình luận sau chuẩn hóa")
plt.show()

### 6.2 Chữ và cụm 2 chữ gặp nhiều nhất

**Ô code làm gì:** in `TOP` chữ và `TOP` cụm 2 chữ liền nhau gặp nhiều nhất trong `text_norm`.

**Đọc kết quả:** đầu bảng thường là chữ chức năng ("không", "là", "thì"). Cụm 2 chữ cho thấy chủ đề rõ hơn: "màn hình", "chụp ảnh", "đáng tiền"…

In [ ]:
TOP = 50
display(pd.concat([
    pd.DataFrame(dem_norm.most_common(TOP), columns=["chữ", "số lần"]),
    pd.DataFrame([(" ".join(k), v) for k, v in dem_cum.most_common(TOP)], columns=["cụm 2 chữ", "số lần "]),
], axis=1))

### 6.3 Ứng viên viết tắt chưa có trong từ điển

**Để làm gì:** tìm cách viết tắt người bình luận hay dùng mà từ điển chưa có, để nhóm quyết có thêm vào không.

**Ô code làm gì:** trên mẫu của mục 3, đếm chữ ở bước ngay trước N6 (đã qua N1–N5, chưa mở viết tắt). Ứng viên là chữ chỉ gồm chữ cái không dấu, dài tối đa 4 ký tự. In hai bảng:
- **Bảng A, không có nguyên âm** (`dc`, `ntn`, `mn`…): gần như chắc là viết tắt.
- **Bảng B, có nguyên âm, chưa có trong từ điển:** có thể là viết tắt, tiếng Việt gõ không dấu ("may", "dep"), hoặc tiếng Anh. Phải đọc ví dụ mới biết.

Mỗi chữ kèm tối đa 3 ví dụ (`text_clean`, cắt 80 ký tự). Thêm vào từ điển thì xem lại danh sách "cố ý chưa đưa vào" ở đầu file TSV.

In [ ]:
def _ung_vien(t: str) -> bool:
    return t.isascii() and t.isalpha() and len(t) <= 4

dem_truoc_n6, VD_TU = Counter(), {}
for i, s in truoc_n6.items():
    ds = tach_tu(s)
    dem_truoc_n6.update(ds)
    for t in ds:
        if _ung_vien(t):
            l = VD_TU.setdefault(t, [])
            if len(l) < 3 and i not in l:
                l.append(i)

def _vi_du(t: str) -> str:
    return " | ".join(df.loc[VD_TU.get(t, []), "text_clean"].str.replace("\n", " ", regex=False).str.slice(0, 80))

ung_vien = pd.DataFrame([(t, n) for t, n in dem_truoc_n6.items() if _ung_vien(t)], columns=["chữ", "số lần"])
ung_vien["có trong từ điển"] = ung_vien["chữ"].isin(TU_DIEN)
co_nguyen_am = ung_vien["chữ"].map(lambda t: bool(set(t) & set("aeiouy")))

bang_a = ung_vien[~co_nguyen_am].sort_values("số lần", ascending=False).head(N_SHOW)
bang_b = ung_vien[co_nguyen_am & ~ung_vien["có trong từ điển"]].sort_values("số lần", ascending=False).head(N_SHOW)
for ten, bang in (("A. Không có nguyên âm", bang_a), ("B. Có nguyên âm, chưa có trong từ điển", bang_b)):
    print(f"\nBảng {ten}:")
    display(bang.assign(**{"ví dụ": bang["chữ"].map(_vi_du)}))
NGUONG_UV = max(1, round(100 * len(mau) / max(int(co_nd.sum()), 1)))   # tương đương 100 lần trên toàn bộ dữ liệu
KQ["Ứng viên viết tắt không nguyên âm, chưa có trong từ điển, ước ≥ 100 lần trên toàn bộ"] = int(
    ((~co_nguyen_am) & ~ung_vien["có trong từ điển"] & (ung_vien["số lần"] >= NGUONG_UV)).sum())

### 6.4 Kiểu đặt dấu thanh

**Để làm gì:** chọn `KIEU_DAU` cho bước N3 theo dữ liệu thật thay vì đoán.

**Ô code làm gì:** trên `text_clean` viết thường (trước N3), đếm số lượt chữ có vần mở `oa`, `oe`, `uy` viết theo kiểu cũ (`hòa`) và kiểu mới (`hoà`), kèm các chữ gặp nhiều nhất của mỗi kiểu.

**Đọc kết quả:** kiểu nào nhiều hơn hẳn thì dùng kiểu đó; đổi trong `normalize.py` rồi tăng `PIPELINE_VERSION`.

In [ ]:
kieu = {"cu": Counter(), "moi": Counter()}
for t, n in dem_clean.items():
    k = L2.kieu_dau_cua_tu(t)
    if k:
        kieu[k][t] = n
tong_cu, tong_moi = sum(kieu["cu"].values()), sum(kieu["moi"].values())
KQ["Lượt chữ đặt dấu kiểu cũ (hòa) / kiểu mới (hoà)"] = f"{tong_cu:,} / {tong_moi:,}"
print(f"Kiểu cũ (hòa): {tong_cu:,} lượt | kiểu mới (hoà): {tong_moi:,} lượt | N3 đang đưa về: {L2.KIEU_DAU}")
display(pd.concat([pd.DataFrame(kieu["cu"].most_common(20), columns=["kiểu cũ", "số lần"]),
                   pd.DataFrame(kieu["moi"].most_common(20), columns=["kiểu mới", "số lần "])], axis=1))

### 6.5 Từ điển viết tắt được dùng bao nhiêu

**Ô code làm gì:** với mỗi mục trong từ điển, đếm số lần chữ đó xuất hiện ở bước ngay trước N6, trên mẫu của mục 3; cột cuối ước tính cho toàn bộ dữ liệu.

**Đọc kết quả:**
- Mục gặp 0 lần thì bỏ được.
- Con số của `k`, `kg`, `v`, `ms` gồm cả những lần đứng sau số. Những lần đó N6 không đổi, nên số lần thực sự được mở ít hơn.

In [ ]:
bang_td = (pd.DataFrame([(k, v, dem_truoc_n6.get(k, 0)) for k, v in TU_DIEN.items()],
                        columns=["viết tắt", "đầy đủ", "số lần gặp"])
             .sort_values("số lần gặp", ascending=False).reset_index(drop=True))
bang_td["ước tính toàn bộ"] = (bang_td["số lần gặp"] * max(int(co_nd.sum()), 1) / len(mau)).round().astype(int)
KQ["Mục từ điển viết tắt không gặp lần nào"] = int((bang_td["số lần gặp"] == 0).sum())
display(bang_td)

## 7. Tổng hợp số liệu

**Đọc kết quả:** các con số chính của notebook này trên một bảng, để chép vào báo cáo `docs/2-tien-xu-ly/bao-cao.md` mục 3.2, 3.5.

In [ ]:
display(pd.Series(KQ, name="giá trị").astype(str).to_frame())